# ETL v1dd 04 - r84 cell-type cluster colors

Populate `Cluster.hex_color` for hierarchy `v1dd_cell_types_r84` using the mapping in `em_cell_type_colors_csm_202610.rtf`. Colors are defined at the parquet `subclass` level; source `cluster` leaves inherit their parent subclass color except `L6b`, which has its own mapping.

In [2]:
import re
from pathlib import Path

import pandas as pd
from deltalake import DeltaTable

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import write_models
from connects_common_connectivity.models import Cluster

In [3]:
OUTPUT_ROOT = get_settings().output_root
HIERARCHY_ID = "EM_csm_cell_types"
COLORS_PATH = Path(
    "/data/v1dd_csm_celltypes_202610/em_cell_type_colors_csm_202610.rtf"
)
CLUSTER_SUBDIR = "cluster"

cluster_table_path = Path(OUTPUT_ROOT) / CLUSTER_SUBDIR
print(f"output_root       : {OUTPUT_ROOT}")
print(f"cluster table path: {cluster_table_path}")
print(f"color mapping     : {COLORS_PATH}")

output_root       : /scratch/v1dd_ccm_20261012
cluster table path: /scratch/v1dd_ccm_20261012/cluster
color mapping     : /data/v1dd_csm_celltypes_202610/em_cell_type_colors_csm_202610.rtf


## Load the supplied mapping

Extract the quoted cell-type and hex-color pairs from the supplied RTF.

In [4]:
rtf_text = COLORS_PATH.read_text(encoding="cp1252")
cell_type_palette_hex = dict(
    re.findall(r'"([^"]+)"\s*:\s*"(#[0-9A-Fa-f]{6})"', rtf_text)
)
assert cell_type_palette_hex, f"No color mappings parsed from {COLORS_PATH}"
assert len(cell_type_palette_hex) == 14
cell_type_palette_hex

{'L2': '#E44919',
 'L3': '#9B2A03',
 'L4': '#35B9BD',
 'L5IT': '#6B63AE',
 'L5ET': '#99ABFE',
 'L5NP': '#871F3D',
 'L6IT': '#8FCE1C',
 'L6CT': '#47863D',
 'L6b': '#40CD9B',
 'NeurogliaformFamily': '#B6A55C',
 'MartinottiFamily': '#07519D',
 'BasketFamily': '#9F5173',
 'Bipolarfamily': '#8472FE',
 'Chandelier': '#B871A2'}

## Read the existing clusters

Read the `Cluster` rows for this hierarchy exactly as written by notebook 01,
so colors are attached to the cell types that actually exist.

In [5]:
cluster_df = DeltaTable(str(cluster_table_path)).to_pandas()
cluster_df = cluster_df[
    cluster_df["hierarchy_id"] == HIERARCHY_ID
].reset_index(drop=True)
print(f"clusters in hierarchy {HIERARCHY_ID!r}: {len(cluster_df)}")
cluster_df[["id", "parent", "level", "hierarchy_category"]].sort_values(
    ["level", "id"]
)

clusters in hierarchy 'EM_csm_cell_types': 149


,id,parent,level,hierarchy_category
0,neuron,None,0,None
1,exc,neuron,1,Class
2,inh,neuron,1,Class
3,BasketFamily,inh,2,Sub-class
4,BipolarFamily,inh,2,Sub-class
...,...,...,...,...
144,NeurogliaformFamily:NeurogliaformFamily-f,NeurogliaformFamily,3,Cluster
145,NeurogliaformFamily:NeurogliaformFamily-g,NeurogliaformFamily,3,Cluster
146,NeurogliaformFamily:NeurogliaformFamily-h,NeurogliaformFamily,3,Cluster
147,NeurogliaformFamily:NeurogliaformFamily-i,NeurogliaformFamily,3,Cluster


## Map hierarchy nodes to colors

`Sub-class` nodes map directly to the RTF keys, with aliases for `L5` -> `L5IT` and `BipolarFamily` -> `Bipolarfamily`. `Cluster` leaves inherit their parent sub-class color, except source cluster `L6b`, which uses the explicit `L6b` entry. Root and `Class` nodes remain uncolored because the supplied mapping does not define them.

In [6]:
SOURCE_TO_PALETTE = {
    "L5": "L5IT",
    "BipolarFamily": "Bipolarfamily",
}
used_palette_keys = set()

def palette_key(source_name):
    return SOURCE_TO_PALETTE.get(source_name, source_name)

def resolve_color(row):
    category = row["hierarchy_category"]
    if category == "Sub-class":
        key = palette_key(row["id"])
    elif category == "Cluster":
        source_cluster = row["id"].split(":", maxsplit=1)[1]
        key = "L6b" if source_cluster == "L6b" else palette_key(row["parent"])
    else:
        return None, None

    color = cell_type_palette_hex.get(key)
    if color is not None:
        used_palette_keys.add(key)
    return color, key if color is not None else None

cluster_df["hex_color"], cluster_df["palette_key"] = zip(
    *cluster_df.apply(resolve_color, axis=1)
)
cluster_df[[
    "id", "level", "hierarchy_category", "palette_key", "hex_color"
]].sort_values(["level", "id"])

,id,level,hierarchy_category,palette_key,hex_color
0,neuron,0,None,None,None
1,exc,1,Class,None,None
2,inh,1,Class,None,None
3,BasketFamily,2,Sub-class,BasketFamily,#9F5173
4,BipolarFamily,2,Sub-class,Bipolarfamily,#8472FE
...,...,...,...,...,...
144,NeurogliaformFamily:NeurogliaformFamily-f,3,Cluster,NeurogliaformFamily,#B6A55C
145,NeurogliaformFamily:NeurogliaformFamily-g,3,Cluster,NeurogliaformFamily,#B6A55C
146,NeurogliaformFamily:NeurogliaformFamily-h,3,Cluster,NeurogliaformFamily,#B6A55C
147,NeurogliaformFamily:NeurogliaformFamily-i,3,Cluster,NeurogliaformFamily,#B6A55C


### Coverage report

Only the root and `Class` nodes are expected to remain uncolored. Any uncolored `Sub-class` or `Cluster` node is an error.

In [7]:
expected_uncolored = set(
    cluster_df.loc[
        cluster_df["hierarchy_category"].isna()
        | (cluster_df["hierarchy_category"] == "Class"),
        "id",
    ]
)
uncolored = set(cluster_df.loc[cluster_df["hex_color"].isna(), "id"])
unexpected_uncolored = sorted(uncolored - expected_uncolored)
unexpected_colored = sorted(expected_uncolored - uncolored)
unused_palette = sorted(set(cell_type_palette_hex) - used_palette_keys)

print("Expected uncolored:", sorted(expected_uncolored))
print("Unexpected uncolored:", unexpected_uncolored)
print("Unexpected colored:", unexpected_colored)
print("Unused palette keys:", unused_palette)

assert not unexpected_uncolored
assert not unexpected_colored
assert not unused_palette

Expected uncolored: ['exc', 'inh', 'neuron']
Unexpected uncolored: []
Unexpected colored: []
Unused palette keys: []


## Write the colored clusters back

Reconstruct `Cluster` models from the existing rows (preserving `parent`,
`children`, and `level`) with `hex_color` populated, then write. The Cluster
write is overwrite-scoped by `hierarchy_id`, so this replaces the hierarchy's
rows in place.

In [8]:
def to_children(value):
    if value is None:
        return None
    items = list(value)
    return [str(item) for item in items] if items else None

def to_opt_int(value):
    return None if pd.isna(value) else int(value)

def to_opt_str(value):
    return None if value is None or pd.isna(value) else str(value)

colored_clusters = [
    Cluster(
        id=str(row.id),
        hierarchy_id=HIERARCHY_ID,
        parent=to_opt_str(row.parent),
        children=to_children(row.children),
        level=to_opt_int(row.level),
        hex_color=to_opt_str(row.hex_color),
        hierarchy_category=to_opt_str(row.hierarchy_category),
    )
    for row in cluster_df.itertuples()
 ]

result = write_models(colored_clusters, output_root=OUTPUT_ROOT)
n_colored = sum(cluster.hex_color is not None for cluster in colored_clusters)
print(
    f"Cluster rows written: {result.rows_written} "
    f"({n_colored}/{len(colored_clusters)} with a hex color)"
)

Cluster rows written: 146 (146/149 with a hex color)


## Verify

Re-read the cluster table and confirm the colors landed.

In [9]:
verify_df = DeltaTable(str(cluster_table_path)).to_pandas()
verify_df = verify_df[verify_df["hierarchy_id"] == HIERARCHY_ID]
verify_df[[
    "id", "level", "hierarchy_category", "parent", "hex_color"
]].sort_values(["level", "id"]).reset_index(drop=True)

,id,level,hierarchy_category,parent,hex_color
0,neuron,0,None,None,None
1,exc,1,Class,neuron,None
2,inh,1,Class,neuron,None
3,BasketFamily,2,Sub-class,inh,#9F5173
4,BipolarFamily,2,Sub-class,inh,#8472FE
...,...,...,...,...,...
144,NeurogliaformFamily:NeurogliaformFamily-f,3,Cluster,NeurogliaformFamily,#B6A55C
145,NeurogliaformFamily:NeurogliaformFamily-g,3,Cluster,NeurogliaformFamily,#B6A55C
146,NeurogliaformFamily:NeurogliaformFamily-h,3,Cluster,NeurogliaformFamily,#B6A55C
147,NeurogliaformFamily:NeurogliaformFamily-i,3,Cluster,NeurogliaformFamily,#B6A55C
